# 01 — Data exploration and profanity candidate search
Kazakh Profanity Detection · DSSC5113 bonus project

Steps:
1. Download of KazSAnDRA (real Kazakh reviews).
2. Basic statistics.
3. Text normalization, so that spelling variants share one form.
4. Keyword search for profanity candidates.
5. Construction of the labeling sheet `labeling_sheet.csv` for manual labeling.
6. Merge of manual labels into `data/labeled.csv`.

In [74]:
import os, sys, re, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 150)


# Colab: clone the repository first
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone https://github.com/<username>/kazakh-profanity-detection.git
    %cd kazakh-profanity-detection
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

## 1. Data download
Only the polarity files are used: train (imbalanced), valid and test. The `ros`/`rus` files are re-sampled copies of the same reviews and are skipped.

In [75]:
if not os.path.exists("KazSAnDRA"):
    !git clone -q https://github.com/IS2AI/KazSAnDRA.git
DATA_DIR = "KazSAnDRA/dataset"
for f in ["01_pc_train_ib", "04_pc_valid", "05_pc_test"]:
    if not os.path.exists(f"{DATA_DIR}/{f}.csv"):
        !unzip -o -q {DATA_DIR}/{f}.zip -d {DATA_DIR}
!ls {DATA_DIR}

01_pc_train_ib.csv   04_pc_valid.csv  06_sc_train_ib.zip   10_sc_test.zip
01_pc_train_ib.zip   04_pc_valid.zip  07_sc_train_ros.zip
02_pc_train_ros.zip  05_pc_test.csv   08_sc_train_rus.zip
03_pc_train_rus.zip  05_pc_test.zip   09_sc_valid.zip


In [76]:
files = ["01_pc_train_ib.csv", "04_pc_valid.csv", "05_pc_test.csv"]
df = pd.concat([pd.read_csv(f"{DATA_DIR}/{f}") for f in files], ignore_index=True)
df = df[["custom_id", "text", "label", "domain"]].rename(columns={"label": "sentiment"})  # 0 = negative, 1 = positive review
print("Rows:", len(df))
df.head()

Rows: 167961


,custom_id,text,sentiment,domain
0,pla015439pla,Өтте күшті,1,appstore
1,pla083193pla,Мәбазар жок .Оте керемет тамаша керемет,1,appstore
2,pla113624pla,"Кушти , дал тура айтады 👍👍👍👍",1,appstore
3,pla029825pla,Реклама коп,0,appstore
4,pla002604pla,5-баға беремін,1,appstore


## 2. Basic statistics

In [77]:
print("Exact duplicate texts:", df.text.duplicated().sum())
df = df.drop_duplicates("text").reset_index(drop=True)
print("After removing duplicates:", len(df))

print("\nReviews per domain:"); print(df.domain.value_counts())
print("\nSentiment (0 = negative, 1 = positive):"); print(df.sentiment.value_counts())
print("\nText length (characters):"); print(df.text.str.len().describe().round(1))

Exact duplicate texts: 1384
After removing duplicates: 166577

Reviews per domain:
domain
appstore     125701
market        28047
mapping        8085
bookstore      4744
Name: count, dtype: int64

Sentiment (0 = negative, 1 = positive):
sentiment
1    137178
0     29399
Name: count, dtype: int64

Text length (characters):
count    166577.0
mean         60.2
std          67.1
min           2.0
25%          24.0
50%          40.0
75%          70.0
max        1939.0
Name: text, dtype: float64


In [78]:
# Look at random reviews from each domain
for d in df.domain.unique():
    print(f"--- {d} ---")
    for t in df[df.domain == d].sample(5, random_state=SEED).text:
        print("  ", t[:120])

--- appstore ---
   Оте керемет алга Казахстан
   М Керемет ойын деген оидамын
   керемет. тиімді. басқа банктерге қарағанда пайыз сомасы төмен
   Маган қаты кызык 👍👍👍
   маған ұнады 5 бал берем😄😄😄
--- market ---
   Ыңғайлы, көлемі жақсы, ұнады.Ондай қатты салқындатпайды. Қымбатырағын алуым керек еді.Жұмысы орташа!
   Өте керемет, рахмет. Қатты ұнады.
   Жақсы тоңазытқыш. Әзірге ешқандай кемшілік байқалмады
   Газ керемет екен, алуға кеңес беремін, супер, тез жеткізіп береді.
   Жаксы аппарат, сварка, болгарка жақсы көтереді.Кемшілік болмады әлі, өз бағасына сай жақсы.Жеткізу тез болды, рахмет.
--- mapping ---
   Ассалаумағалайкум
   Керемет демалыс орны. Жастарға да үлкен кісілерге де жайлы жер.
   Ең әдемі және тамаша мұражай
   Өте керемет фитнес клуб. Өзіме қатты ұнады. Әсіресе тек қана әел адамдарға арналғаны. Тренерлері де тамаша қыздар. Әсіре
   Рахмет сіздерге барлығы дамді Айгерим официант кушти кыз
--- bookstore ---
   Рахмет емес Рақмет
   Неге ақылы болып кеткен? алдында ға

## 3. Text normalization
The same word appears in many spellings: `қотақ`, `котак`, `КОТАААК`, `kotak`.
`normalize()` maps them to one form, so a single search pattern covers all variants, defined in `src/preprocess.py`.

In [79]:
from src.preprocess import normalize

for s in ["ҚОТАААҚ", "Сапасы өте нашар!!!", "kotak bas"]:
    print(s, "->", normalize(s))

df["norm"] = df.text.map(normalize)

ҚОТАААҚ -> котак
Сапасы өте нашар!!! -> сапасы оте нашар!
kotak bas -> котак бас


## 4. Profanity candidate search
Profane roots are listed in **normalized** form, Kazakh and Russian (Russian swear words count as profanity under the labeling rules).
A match does **not** mean the review is profane; it only marks the review for manual checking.

Mild insults (мал, ешек, ақымақ, жынды) are not in the list: under the labeling rules they are label 0.

In [80]:
ROOTS = {
    # Kazakh
    "сиг":     r"\bс[иы]?г[еи]|\bс[иы]г[иы]л|\bс[иы]гис|\bс[иы]кпе|\bс[иы]кт[иы]р|\bс[иы]кти\b|\bс[иы]кен|\bс[иы]ктим",
    "котак":   r"кота[кг]",
    "ам":      r"\bамы\b|\bамын|\bамына\b|\bамдар|\bамшык|\bам жалаг",
    "шешенди": r"шешен[дт]и(?!к)",
    "кот":     r"\bкоти[нм]|\bкотке\b",
    "бок":     r"\bбок\b|\bбоктар|\bбокка|\bбокты\b",
    "жезокше": r"жезокше",
    "каншык":  r"кан[шч]ык",
    "саикал":  r"саикал",
    "жалап":   r"\bж[аә]л[ае]п|салдакы",
    "ташак":   r"ташак",
    "шшс":     r"\b[шщ][шщ]с\b|\b667\b|\bшешес\b|\bшешеси\b",
    # Russian
    "ебтвоюмать": r"\bе[бп]\s?т\w{0,6}\s?ма[иі]?т|\bе[бп]т[аы]?\b",
    "хуй":     r"ху[иеяю]|нахуи|\bнах[ук]\b",
    "пизд":    r"пизд|пезд",
    "еб":      r"\bеба|\bебан|\bебл|\bебу|\bеби|\bеб\b|заеб|уеб|наеб|выеб",
    "долбаеб": r"д[оа]лб[оа][еи]|д[оа]лб[оа]и?[оа]п|заибал|заебал",
    "бля":     r"\bбля|\bбл[аи]а?[тд]",
    "сука":    r"\bсук[аи]\b|\bсуч",
    "пидор":   r"пид[оа]?р|педик(?![юу]р)",
    "гандон":  r"\bг[оа]нд[оа]н",
    "шлюх":    r"шлюх",
    "говно":   r"говн|дерьм"
}
MASK = r"[а-яa-zәіңғүұқөһ][\*#\$]+[а-яa-zәіңғүұқөһ]"  # masked words, e.g. с**ейін, бл*ть

def matched_roots(row):
    found = [k for k, p in ROOTS.items() if re.search(p, row)]
    return ",".join(found)

df["matched"] = df.norm.map(matched_roots)
df.loc[df.text.str.lower().str.contains(MASK, regex=True), "matched"] += ",mask"
df["matched"] = df.matched.str.strip(",")
df["candidate"] = df.matched != ""

print("Candidates:", df.candidate.sum(), "of", len(df))

Candidates: 2455 of 166577


In [81]:
# How many matches per root
counts = {k: df.matched.str.split(",").map(lambda x: k in x).sum() for k in list(ROOTS) + ["mask"]}
pd.Series(counts).sort_values(ascending=False)

,0
котак,1048
долбаеб,359
ам,286
бля,216
сиг,209
хуй,194
еб,181
шшс,178
пизд,155
сука,115


In [82]:
# Where do the candidates come from?
print(pd.crosstab(df[df.candidate].domain, df[df.candidate].sentiment, margins=True))
# Many candidates are POSITIVE reviews: profanity is not the same as a negative review.

sentiment     0     1   All
domain                     
appstore   1308  1122  2430
bookstore     4     7    11
mapping       9     1    10
market        1     3     4
All        1322  1133  2455


In [83]:
# Examples for one root
def show(root, n=15):
    sub = df[df.matched.str.split(",").map(lambda x: root in x)]
    print(f"{root}: {len(sub)} matches")
    for t in sub.sample(min(n, len(sub)), random_state=SEED).text:
        print("  •", t[:140])

show("котак")

котак: 1048 matches
  • Қотағыма кигенім ризен гандон халқым ай Қотағымнан барады ызғар отіп қалқым ай
  • Версия Pro қотақ жеген тегңн неге істемеиды паидаланып қалуғо пидараздырға мал сука Мұрат Әбділдәға айтам маған поху
  • Қотақьас блять далбаепко Реклама дахуя қотақбас амшық, зайбал видео тола ішінде қотақбас, амдары сасыған қыздар көпкош шешең амы, сіккің кел
  • Адамды жынды қылып жіберді шешең амыскіині қотак бас
  • Мал кірілмиді қотақ бас
  • Нұрдәулет мал ешак қотақ амшық
  • Котагм
  • Қотақбайский
  • Е шешен амы котактар сап-сары котак шешен амы салбыраған сасып кеткен котағын шешесігейің котак кет неху котағымды жемей мусырный жәшк кусам
  • Қотақбас амыңа кірсін
  • Шешен дал амы шешен амыссыгин котактыда кирмид блядь сигилип кеттикко шшессгин 2 сагат болдым сука пароль алуга котакбастар шыгарсандар норм
  • Котагыма отирик
  • 🖕🖕🖕🖕🖕🖕🖕🖕🖕🖕🖕🖕🖕🖕сүй қотақғымның басы сігіліб кет нау вирус бар !!!!шешеннің амы
  • Ғалымжан қотақбас мал гей
  • Қазақстан тілі жоқ мал қотақ


In [84]:
show("кот")

кот: 71 matches
  • Ашумен келдім Астанаға дейін. Именно 103 нөмерлі сары автобусқа мінбеңдер. Ақшадан өлген, құныққан Бекжан автовогзал. Админстрация қадағалау
  • Өте керемет көтінді қйнайды брақ
  • Рахман лекер істи алмасаң көтіңді қысып тыныш оста
  • Кіргізема көтің қонаққа
  • Мен бұл ойынды байағдан ойнап жүргем ратушам 8 болатын сосын бір далбаёб жойып сапты жойған адамның көтін айырдым одаң кейін бірінші ратушад
  • Футболыңыз керемет екен не пиздессіңбане көтіңді қыс наху к қ не пиздесба не ойын
  • котимди суй
  • Қотақ бас маған бас пер көтінқышы
  • Ма котине тык
  • Көтіңді қысып жүр !
  • Жауапкершілігі өте төмен, барлық пойыз кешігіп келеді. Бұлай жердің көтіне вокзал салмай ақ қойғаны жақсы еді. 
  • Автор көтіңді сгеин сенің кесірінен мама папа ұрды ыыы
  • Котактын ойыны амынды сігейін бе сорып жане котагымды котине тыгайын ба келде торгайдын жеміне сігіл маган лайк басындар я
  • көтім тырысып калд рулар жок
  • Маған бұл оиын өте қаты ұнамады көтіңді қыз білдінба 

In [85]:
show("mask")

mask: 50 matches
  • акшамды жеп койд е*шак жазылгамго бир футбол коре салайн деп м*л
  • Бәрі күшті бірақ қазақтардың көбі орысса жазған қ***қ бастар екен. Мен орысша білемін бірақ өтініш орысша жазбаныздаршы айтубта. Қазақша жаз
  • Че за қыртыған ойын ? Самый не логичный, ақымақ ойын, где ойыншылардың 90 пайызы ойынды сразу жалғастырмайды, вообще не үшін 12 сағат береді
  • Шшс қате қате ғо мынауың е сауатсыздар оқып алып шығарыңдар программаларыңды соттап түрмеге тығу керек сендердей сауатсыздарды программа нол
  • Маған ойын ұнады......$$б$#б$
  • Миллион доллар с**а
  • Кете есек мал кот*қ бұл предложения керемет
  • Прям айта алмадым элско ғаоышқа галактика жаратылысқа қызығатвн адамдар оқысын п*здец
  • Ё*тваймач басымды қатырды сөздерді табалмай есім шықты с*ка
  • Қота*бас клаш аккымды қаитарып бер нахуй зб
  • Кірмейді не*уя
  • ешекбас бл*ть
  • Жарнама жарнама Қ****бас
  • Күшті бірақ қата берет😡 б#ә
  • Бұндай обновлениелеріңіздің шешесін с**ейін😡. Обновлениеден кейін мар

## 5. Hard negatives
Rude or angry reviews **without** profanity. These teach the model that rudeness alone is label 0.

In [86]:
MILD = r"\bмал\b|\bмалдар|ешек|акымак|жынды|надан|\bит\b|хаиуан|жаман"
df["mild"] = df.norm.str.contains(MILD, regex=True) & ~df.candidate
print("Mild / angry non-candidates:", df.mild.sum())
for t in df[df.mild].sample(10, random_state=SEED).text:
    print("  •", t[:120])

Mild / angry non-candidates: 4551
  • Осы VK не үшін керек осы қуып кеткен группалар бар мен өз басым вкка кірмейм мал вка
  • 😡😡😡😡өте жаман бұл ойынды тартып алмаңыздар🙅‍♀️ ваше ашылмат😡😡😡
  • Тфуу мә ффдан күшті дейді ғо білмесен если что пубгта харам заттар көп общем жаман ойын👎 фф хорош👍
  • Кажети жокы жаман ашпаит нервгатиып отыр канды ишти
  • Амал жок колдану керек а так жаман куад
  • Жаман бір жұлдыз 🙅‍♀️🙅‍♀️🙅‍♀️🙅‍♀️🙅‍♀️🙅‍♀️🙅‍♀️🙅‍♀️
  • Малдар.Неге мен бағамды қарау үшін ақша төлей керекпін?Мектепте оқитын балада подписка алу үшін ақша болама?
  • Бұрыннан ойнаймын тағы тартып жатырмын керемет ойын🙃 жаңадан ойын қоссаңдар тіптен жақсы болар еді😊 сұраныста көбейеді
  • Сабын өте жақсы. Бетті жұмсартып, жаңадан шығып келе жатқан прищиктерді кетіреді. Рақмет Флип!
  • Жаксы немесе жаман😂😂😂😂


## 6. Labeling sheet
The sheet contains:
- **all** candidates (each one checked manually)
- **hard negatives**: rude but clean reviews
- a **random sample** of other reviews from every domain, half negative and half positive

`suggested_label` is only a hint (1 for candidates); the final `label` is assigned manually.

In [87]:
def remove_personal_info(t):
    t = re.sub(r"\+?\d[\d\s\-()]{8,}\d", "[PHONE]", t)
    t = re.sub(r"\S+@\S+\.\S+", "[EMAIL]", t)
    t = re.sub(r"@\w+", "[USER]", t)
    return t

rest = df[~df.candidate & ~df.mild]
parts = [
    df[df.candidate].assign(group="candidate"),
    df[df.mild].sample(min(600, df.mild.sum()), random_state=SEED).assign(group="hard_negative"),
]
# random clean-looking reviews: per domain, half negative / half positive
for dom, n in {"appstore": 1200, "market": 1200, "mapping": 300, "bookstore": 300}.items():
    for s in [0, 1]:
        pool = rest[(rest.domain == dom) & (rest.sentiment == s)]
        parts.append(pool.sample(min(n // 2, len(pool)), random_state=SEED).assign(group="random"))

sheet = pd.concat(parts, ignore_index=True)
sheet["text"] = sheet.text.map(remove_personal_info)
sheet["suggested_label"] = (sheet.group == "candidate").astype(int)
sheet["label"] = ""          # filled manually: 0 or 1
sheet["uncertain"] = ""      # 1 if the label is unclear
sheet["synthetic"] = 0
sheet["source"] = "kazsandra"
sheet["note"] = ""
sheet = sheet.sample(frac=1, random_state=SEED)                          # shuffle
sheet = sheet.sort_values("group", key=lambda g: g != "candidate")      # candidates first
cols = ["custom_id", "text", "domain", "sentiment", "group", "matched",
        "suggested_label", "label", "uncertain", "synthetic", "source", "note"]
sheet[cols].to_csv("labeling_sheet.csv", index=False)

print("Rows to label:", len(sheet))
print(sheet.group.value_counts())

Rows to label: 6055
group
random           3000
candidate        2455
hard_negative     600
Name: count, dtype: int64


## 7. Merge of manual labels
Manual labeling was done following the labeling rules in the README:
- all 2,455 candidates and 600 hard negatives were checked manually;
- 300 of the 3,000 random reviews were checked manually (1 contained profanity missed by the keyword list);
- the remaining 2,700 random reviews are labeled 0 without manual checking (`label_method = assumed_0`) and are used only for training and validation.

Output: `data/labeled.csv`, the input for notebook 02.

In [89]:
lab = pd.read_csv("manual_labels.csv").drop_duplicates("custom_id")   # manually labeled rows
lab["label_method"] = "manual"

# remaining random rows, labeled 0 without manual checking
extra = sheet[(sheet.group == "random") & ~sheet.custom_id.isin(lab.custom_id)].copy()
extra[["label", "uncertain", "label_method"]] = [0, 0, "assumed_0"]

out = pd.concat([lab, extra], ignore_index=True)[cols + ["label_method"]]
out["label"] = pd.to_numeric(out["label"], errors="coerce").astype("Int64")
out["uncertain"] = pd.to_numeric(out["uncertain"], errors="coerce").fillna(0).astype("Int64")

os.makedirs("data", exist_ok=True)
out.to_csv("data/labeled.csv", index=False)
print(out.label_method.value_counts(), out.label.value_counts(), sep="\n")

label_method
manual       3355
assumed_0    2700
Name: count, dtype: int64
label
0    3598
1    2457
Name: count, dtype: Int64
